# Assignment 2: Transformer language models

Compact solution notebook. Tasks are separated, and 🎓 tasks include a short why/how note.


## Step 0: Preliminaries


In [1]:
from pathlib import Path
from collections import Counter
import math
import random

import nltk
import torch
from torch import nn
import torch.nn.functional as F
from torch.distributions import Categorical
from torch.utils.data import DataLoader, Subset
from datasets import load_dataset
from transformers import BatchEncoding, PretrainedConfig, PreTrainedModel, TrainingArguments
from transformers.modeling_outputs import CausalLMOutput

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


def get_device(use_cpu=False):
    if use_cpu:
        return torch.device('cpu')
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


device = get_device()
print('device:', device)


/Users/telio/miniconda3/envs/phenoVLM-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


## Dataset And Assignment 1 Utilities

Assignment 2 reuses the Assignment 1 text data, word splitting, vocabulary construction, tokenizer, batching style, and training-loop shape. The only major replacement is the model: the RNN language model becomes a Transformer language model.


In [2]:
A1_DATA_URL = 'https://www.cse.chalmers.se/~richajo/waspnlp2026/a1_1.zip'
DATA_ROOT = Path('data')
A1_DATA_DIR = DATA_ROOT / 'a1_1'
ARCHIVE_FILE = DATA_ROOT / 'a1_1.zip'
TRAIN_FILE = A1_DATA_DIR / 'train.txt'
VAL_FILE = A1_DATA_DIR / 'val.txt'
HF_DATASETS_CACHE = DATA_ROOT / 'hf_datasets_cache'

MAX_VOC_SIZE = 20_000
MODEL_MAX_LENGTH = 80
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
BOS_TOKEN = '<BOS>'
EOS_TOKEN = '<EOS>'


def ensure_assignment_data():
    if TRAIN_FILE.exists() and VAL_FILE.exists():
        return

    from urllib.request import urlretrieve
    import zipfile

    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    print('Downloading Assignment 1 data...')
    urlretrieve(A1_DATA_URL, ARCHIVE_FILE)
    print('Extracting Assignment 1 data...')
    with zipfile.ZipFile(ARCHIVE_FILE) as archive:
        archive.extractall(DATA_ROOT)

    assert TRAIN_FILE.exists(), f'Missing expected file after extraction: {TRAIN_FILE}'
    assert VAL_FILE.exists(), f'Missing expected file after extraction: {VAL_FILE}'


def lowercase_tokenizer(text):
    return [token.lower() for token in nltk.word_tokenize(text)]


def read_nonempty_lines(path):
    with path.open(encoding='utf-8', errors='ignore') as handle:
        return [line.strip() for line in handle if line.strip()]


def build_vocabulary(tokenized_texts, max_voc_size=None):
    special_tokens = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]
    if max_voc_size is not None and max_voc_size < len(special_tokens):
        raise ValueError('max_voc_size must leave room for the four special tokens')

    counts = Counter(token for text in tokenized_texts for token in text)
    max_words = None if max_voc_size is None else max_voc_size - len(special_tokens)
    words = [word for word, _ in counts.most_common(max_words) if word not in special_tokens]
    int_to_str = special_tokens + words
    str_to_int = {word: idx for idx, word in enumerate(int_to_str)}
    return str_to_int, int_to_str


class A1Tokenizer:
    """Assignment 1 tokenizer reused for Assignment 2."""

    def __init__(self, str_to_int, int_to_str, tokenize_fun=lowercase_tokenizer, model_max_length=None):
        self.str_to_int = str_to_int
        self.int_to_str = int_to_str
        self.vocab = str_to_int
        self.inv_vocab = int_to_str
        self.tokenize_fun = tokenize_fun
        self.model_max_length = model_max_length
        self.pad_token_id = str_to_int[PAD_TOKEN]
        self.unk_token_id = str_to_int[UNK_TOKEN]
        self.bos_token_id = str_to_int[BOS_TOKEN]
        self.eos_token_id = str_to_int[EOS_TOKEN]

    def __len__(self):
        return len(self.int_to_str)

    def encode(self, text, truncation=False, add_eos=True):
        tokens = [BOS_TOKEN] + self.tokenize_fun(text)
        if add_eos:
            tokens.append(EOS_TOKEN)
        ids = [self.str_to_int.get(token, self.unk_token_id) for token in tokens]
        if truncation and self.model_max_length is not None and len(ids) > self.model_max_length:
            ids = ids[:self.model_max_length]
            if add_eos:
                ids[-1] = self.eos_token_id
        return ids

    def decode(self, ids, skip_special_tokens=True):
        words = []
        special = {PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN}
        for idx in ids:
            word = self.int_to_str[int(idx)] if int(idx) < len(self.int_to_str) else UNK_TOKEN
            if skip_special_tokens and word in special:
                continue
            words.append(word)
        return ' '.join(words)

    def __call__(self, texts, truncation=False, padding=False, return_tensors=None):
        if isinstance(texts, str):
            texts = [texts]
        if return_tensors not in {None, 'pt'}:
            raise ValueError("return_tensors must be None or 'pt'")

        encoded = [self.encode(text, truncation=truncation) for text in texts]
        attention_mask = [[1] * len(ids) for ids in encoded]

        if padding:
            max_len = max(len(ids) for ids in encoded)
            encoded = [ids + [self.pad_token_id] * (max_len - len(ids)) for ids in encoded]
            attention_mask = [mask + [0] * (max_len - len(mask)) for mask in attention_mask]

        if return_tensors == 'pt':
            encoded = torch.tensor(encoded, dtype=torch.long)
            attention_mask = torch.tensor(attention_mask, dtype=torch.long)
        return BatchEncoding({'input_ids': encoded, 'attention_mask': attention_mask})


def build_tokenizer(train_file, tokenize_fun=lowercase_tokenizer, max_voc_size=None, model_max_length=None):
    texts = read_nonempty_lines(Path(train_file))
    tokenized = [tokenize_fun(text) for text in texts]
    str_to_int, int_to_str = build_vocabulary(tokenized, max_voc_size=max_voc_size)
    return A1Tokenizer(str_to_int, int_to_str, tokenize_fun=tokenize_fun, model_max_length=model_max_length)


def collate_texts(batch):
    return [example['text'] for example in batch]


def make_lm_batch(texts, tokenizer, device):
    encoded = tokenizer(texts, return_tensors='pt', padding=True, truncation=True)
    input_ids = encoded['input_ids'].to(device)
    labels = input_ids.clone()
    labels[labels == tokenizer.pad_token_id] = -100
    return input_ids, labels


def encode_prompt(text):
    return tokenizer.encode(text, truncation=True, add_eos=False)


ensure_assignment_data()
tokenizer = build_tokenizer(TRAIN_FILE, max_voc_size=MAX_VOC_SIZE, model_max_length=MODEL_MAX_LENGTH)

HF_DATASETS_CACHE.mkdir(parents=True, exist_ok=True)
dataset = load_dataset(
    'text',
    data_files={'train': str(TRAIN_FILE), 'val': str(VAL_FILE)},
    cache_dir=str(HF_DATASETS_CACHE),
)
dataset = dataset.filter(lambda example: example['text'].strip() != '')
print(dataset)

USE_SMALL_DATASET = False
N_TRAIN_EXAMPLES = 2_000
N_VAL_EXAMPLES = 2_000

if USE_SMALL_DATASET:
    dataset['train'] = Subset(dataset['train'], range(min(N_TRAIN_EXAMPLES, len(dataset['train']))))
    dataset['val'] = Subset(dataset['val'], range(min(N_VAL_EXAMPLES, len(dataset['val']))))

print('used split sizes:', {'train': len(dataset['train']), 'validation': len(dataset['val'])})
print('vocab size:', len(tokenizer))


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 147059
    })
    val: Dataset({
        features: ['text'],
        num_rows: 17874
    })
})
used split sizes: {'train': 147059, 'validation': 17874}
vocab size: 20000


## 🎓 Task 1.1: MLP layer

**Why/how.** OLMo-style decoders use a gated feed-forward block. SwiGLU multiplies a value projection by a learned gate, then projects back to the hidden size.


In [3]:
class A2ModelConfig(PretrainedConfig):
    model_type = 'a2-transformer-lm'
    def __init__(self, vocab_size=0, hidden_size=96, intermediate_size=192, num_hidden_layers=2,
                 num_attention_heads=4, max_position_embeddings=128, rms_norm_eps=1e-6, rope_theta=10000.0, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.max_position_embeddings = max_position_embeddings
        self.rms_norm_eps = rms_norm_eps
        self.rope_theta = rope_theta

class SwiGLUMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

cfg = A2ModelConfig(vocab_size=len(tokenizer))
x = torch.randn(2, 5, cfg.hidden_size)
print(SwiGLUMLP(cfg)(x).shape)


torch.Size([2, 5, 96])


## ⚙ Task 1.2: Normalization


In [4]:
class RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.eps = eps
    def forward(self, x):
        x_float = x.float()
        normed = x_float * torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (self.weight * normed).to(dtype=x.dtype)

print(RMSNorm(cfg.hidden_size)(x).shape)

torch.Size([2, 5, 96])


## 🎓 Task 1.3: Multi-head attention

**Why/how.** Causal self-attention lets each token attend only to previous tokens. RoPE injects position information by rotating query/key dimensions before attention scores are computed.


In [5]:
def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
    return (q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)
    def forward(self, seq_len, device):
        t = torch.arange(seq_len, device=device).float()
        freqs = torch.einsum('i,j->ij', t, self.inv_freq.to(device))
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]

class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.hidden_size % config.num_attention_heads == 0
        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.q_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.o_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.q_norm = RMSNorm(self.head_dim, config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, config.rms_norm_eps)
    def _shape(self, x):
        b, m, _ = x.shape
        return x.view(b, m, self.num_heads, self.head_dim).transpose(1, 2)
    def forward(self, hidden_states, rope_rotations):
        b, m, d = hidden_states.shape
        q = self.q_norm(self._shape(self.q_proj(hidden_states)))
        k = self.k_norm(self._shape(self.k_proj(hidden_states)))
        v = self._shape(self.v_proj(hidden_states))
        cos, sin = rope_rotations
        q, k = apply_rope(q, k, cos, sin)
        attn = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn = attn.transpose(1, 2).reshape(b, m, d)
        return self.o_proj(attn)

rope = RotaryEmbedding(cfg.hidden_size // cfg.num_attention_heads)
attn = MultiHeadAttention(cfg)
y = attn(x, rope(seq_len=x.shape[1], device=x.device))
print(y.shape)

torch.Size([2, 5, 96])


## 🎓 Task 1.4: The full Transformer decoder layer

**Why/how.** The decoder layer uses pre-norm residual blocks: normalize, apply attention/MLP, then add the result back. Residual connections keep gradients stable across many layers.


In [6]:
class DecoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_norm = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.self_attn = MultiHeadAttention(config)
        self.post_attention_norm = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.mlp = SwiGLUMLP(config)
    def forward(self, hidden_states, rope_rotations):
        hidden_states = hidden_states + self.self_attn(self.input_norm(hidden_states), rope_rotations)
        hidden_states = hidden_states + self.mlp(self.post_attention_norm(hidden_states))
        return hidden_states

layer = DecoderLayer(cfg)
print(layer(x, rope(seq_len=x.shape[1], device=x.device)).shape)

torch.Size([2, 5, 96])


## ⚙ Task 1.5: The complete Transformer stack


In [7]:
class A2TransformerLM(PreTrainedModel):
    config_class = A2ModelConfig

    def __init__(self, config):
        super().__init__(config)
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([DecoderLayer(config) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.rotary_emb = RotaryEmbedding(config.hidden_size // config.num_attention_heads, config.rope_theta)
        self.loss_func = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, input_ids, labels=None):
        h = self.embed_tokens(input_ids)
        rope = self.rotary_emb(h.shape[1], h.device)
        for layer in self.layers:
            h = layer(h, rope)
        logits = self.lm_head(self.norm(h))
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = self.loss_func(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        return CausalLMOutput(loss=loss, logits=logits)


model = A2TransformerLM(cfg).to(device)
example_texts = [dataset['train'][i]['text'] for i in range(4)]
input_ids, labels = make_lm_batch(example_texts, tokenizer, device)
out = model(input_ids=input_ids, labels=labels)
print(out.logits.shape, float(out.loss))


torch.Size([4, 80, 20000]) 10.081053733825684


## ⚙ Task 2.1: Training the language model

This reuses the Assignment 1 trainer shape with the Transformer model as a drop-in replacement. Change the training knobs at the top of the next cell when you want a longer run or a different batch size.


In [8]:
class A2Trainer:
    """A small Assignment-1-style trainer for the Transformer LM."""

    def __init__(self, model, args, train_dataset, eval_dataset, tokenizer):
        self.model = model
        self.args = args
        self.train_dataset = train_dataset
        self.eval_dataset = eval_dataset
        self.tokenizer = tokenizer
        self.device = get_device(use_cpu=args.use_cpu)

    def _loader(self, dataset, batch_size, shuffle=False):
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_texts)

    def evaluate(self):
        self.model.eval()
        loader = self._loader(self.eval_dataset, self.args.per_device_eval_batch_size)
        losses = []
        with torch.no_grad():
            for texts in loader:
                input_ids, labels = make_lm_batch(texts, self.tokenizer, self.device)
                losses.append(self.model(input_ids=input_ids, labels=labels).loss.item())
        return sum(losses) / max(1, len(losses))

    def train(self):
        self.model.to(self.device)
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.args.learning_rate, weight_decay=0.01)
        train_loader = self._loader(self.train_dataset, self.args.per_device_train_batch_size, shuffle=True)
        history = []

        for epoch in range(1, int(self.args.num_train_epochs) + 1):
            self.model.train()
            total = 0.0
            for texts in train_loader:
                input_ids, labels = make_lm_batch(texts, self.tokenizer, self.device)
                loss = self.model(input_ids=input_ids, labels=labels).loss
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                total += loss.item()

            train_loss = total / max(1, len(train_loader))
            valid_loss = self.evaluate() if self.args.eval_strategy == 'epoch' else float('nan')
            row = {
                'epoch': epoch,
                'train_loss': train_loss,
                'valid_loss': valid_loss,
                'valid_ppl': math.exp(min(valid_loss, 20)),
            }
            history.append(row)
            print(row)

        print(f'Saving to {self.args.output_dir}.')
        self.model.save_pretrained(self.args.output_dir)
        return history


TRAIN_EPOCHS = 3
TRAIN_BATCH_SIZE = 24
EVAL_BATCH_SIZE = 48
LEARNING_RATE = 2e-3
USE_CPU = False
OUTPUT_DIR = f'results/assignments/a2_transformer_epochs{TRAIN_EPOCHS}_bs{TRAIN_BATCH_SIZE}'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    optim='adamw_torch',
    eval_strategy='epoch',
    use_cpu=USE_CPU,
    learning_rate=LEARNING_RATE,
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    report_to=[],
    save_strategy='no',
)

print({
    'epochs': TRAIN_EPOCHS,
    'train_batch_size': TRAIN_BATCH_SIZE,
    'eval_batch_size': EVAL_BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'output_dir': OUTPUT_DIR,
})

trainer = A2Trainer(model, training_args, dataset['train'], dataset['val'], tokenizer)
history = trainer.train()


{'epochs': 3, 'train_batch_size': 24, 'eval_batch_size': 48, 'learning_rate': 0.002, 'output_dir': 'results/assignments/a2_transformer_epochs3_bs24'}
{'epoch': 1, 'train_loss': 5.183970700200482, 'valid_loss': 4.818946904854864, 'valid_ppl': 123.8346124596586}
{'epoch': 2, 'train_loss': 4.683675729263545, 'valid_loss': 4.664840604280978, 'valid_ppl': 106.14866423078591}
{'epoch': 3, 'train_loss': 4.538242403972243, 'valid_loss': 4.607346746940715, 'valid_ppl': 100.2178931380896}
Saving to results/assignments/a2_transformer_epochs3_bs24.


## ⚙ Task 3.1: Predicting the next word


In [9]:
def predict_next(model, prompt, topk=5):
    model.eval()
    ids = torch.tensor([encode_prompt(prompt)], dtype=torch.long, device=trainer.device)
    with torch.no_grad():
        logits = model(ids).logits[0, -1]
        probs = logits.softmax(dim=-1)
        values, indices = probs.topk(topk)
    return [(tokenizer.int_to_str[int(idx)], float(prob)) for idx, prob in zip(indices, values)]


predict_next(model, 'he lives in san')


[('francisco', 0.27233824133872986),
 ('salvador', 0.2694322168827057),
 ('<UNK>', 0.1390417069196701),
 ('diego', 0.10496339946985245),
 ('juan', 0.021670110523700714)]

## 🎓 Task 3.2: Generating texts

**Why/how.** Autoregressive generation repeatedly feeds the current prefix through the model, samples the next token from the final-position logits, appends it, and stops at EOS or a length limit.


In [12]:
@torch.no_grad()
def generate_text(model, prompt, max_length=30, temperature=1.0, topk=10):
    model.eval()
    ids = encode_prompt(prompt)
    for _ in range(max_length):
        x = torch.tensor([ids[-cfg.max_position_embeddings:]], dtype=torch.long, device=trainer.device)
        logits = model(x).logits[0, -1] / max(temperature, 1e-6)
        if topk is not None:
            values, indices = logits.topk(min(topk, logits.numel()))
            next_id = int(indices[Categorical(logits=values).sample()])
        else:
            next_id = int(Categorical(logits=logits).sample())
        ids.append(next_id)
        if next_id == tokenizer.eos_token_id:
            break
    return tokenizer.decode(ids)


for temp in [0.7, 1.0, 1.4]:
    print(temp, '->', generate_text(model, 'in natural language processing, a transformer', temperature=temp, topk=8))


0.7 -> in natural language processing , a , was used in the , a , which the ( ) and the ( ) , (
1.0 -> in natural language processing , a of is used as an example of , which allows the of the to form the language .
1.4 -> in natural language processing , a , and ( also known as ( ) , and ( ) is the of ( ``


## 🎓 Task 3.3: Comparing to a pre-trained Transformer

**Why/how.** The small model is trained on this assignment's Wikipedia subset, so it mostly learns local patterns. A pre-trained OLMo-2 model has learned from broad text data, so its continuations are usually more grammatical and coherent, though it is not necessarily instruction-following.


In [11]:
RUN_PRETRAINED = True
if RUN_PRETRAINED:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    model_name = 'allenai/OLMo-2-0425-1B'
    hf_tokenizer = AutoTokenizer.from_pretrained(model_name)
    hf_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    prompt = 'In natural language processing, a Transformer'
    inputs = hf_tokenizer(prompt, return_tensors='pt').to(device)
    out = hf_model.generate(**inputs, max_new_tokens=50, do_sample=True, top_k=50, temperature=0.8)
    print(hf_tokenizer.decode(out[0], skip_special_tokens=True))

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 22.97it/s]


In natural language processing, a Transformer model usually consists of five encoder and four decoder layers (including the self-attention layer and the feed forward layer). The feed forward layer is also called the transducer, which is responsible for the conversion of words into numbers. In the encoder layer,
